# Following LGALS3 and NLRP3 through OptimusKG

## A progressive tutorial in evidence navigation, entity comparison and graph thinking

This notebook asks:

> What disease, compound, pathway and biological-process context does OptimusKG record around Galectin-3, and how does that evidence landscape compare with NLRP3?

NLRP3 is used as a **reference entity** because it is an established intracellular sensor involved in inflammasome activation by endogenous danger signals and sterile tissue injury. It is not introduced as a newly predicted target.

The notebook progresses through:

1. inspecting the OptimusKG schema;
2. resolving `LGALS3` and `NLRP3` precisely;
3. loading selected relationship layers;
4. auditing direct evidence and provenance;
5. comparing typed neighbourhoods;
6. applying a transparent neighbourhood-overlap measure;
7. identifying representation gaps and future questions.

### What this notebook may establish

- relationships recorded in the pinned OptimusKG release;
- differences in database coverage around two genes;
- shared and unique functional contexts;
- candidate questions for literature or experimental review.

It does **not** establish that either gene causes a disease, that a compound treats a disease, or that a shared annotation is an active causal mechanism.


## Why compare LGALS3 with NLRP3?

Sterile inflammation begins when host-derived signals from damaged or stressed tissue are detected by innate immune machinery. NLRP3 is one well-established component of this response, but its activation routes vary by stimulus, tissue and experimental system.

Published studies provide context-specific evidence connecting Galectin-3 with NLRP3:

- hepatic macrophage experiments and a mouse model of cholestatic injury reported Galectin-3-dependent NLRP3 activation;
- a Huntington's disease model reported Galectin-3-associated microglial inflammation involving NF-kB- and NLRP3-dependent pathways.

Those findings justify a comparison, but they do not make the relationship universal. This notebook asks whether OptimusKG represents the same contexts and where its evidence differs from the selected literature.


## 1. Setup


In [ ]:
# In a fresh Codespace run this once, then restart the kernel:
# %pip install -r requirements.txt

from pathlib import Path
import gc
import math
import re
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx
import polars as pl
from IPython.display import display
import requests
import optimuskg._dataverse as dataverse

# Some hosted environments require an explicit user-agent for Dataverse.
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})
dataverse.requests = session

PROJECT_ROOT = Path.cwd().resolve()

# Support either project/core.py or project/optimus_explorer/core.py.
for candidate in [PROJECT_ROOT, PROJECT_ROOT / "optimus_explorer"]:
    if (candidate / "core.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from core import OptimusExplorer

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lgals3_nlrp3_tutorial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)


## 2. Connect without loading the graph

Pinning the DOI makes the analysis repeatable against the same OptimusKG release. The catalogue shows what may be loaded; it does not download every table.


In [ ]:
OPTIMUSKG_DOI = "doi:10.7910/DVN/IYNGEV"

kg = OptimusExplorer.from_optimuskg(doi=OPTIMUSKG_DOI)
display(kg.status())


In [ ]:
display(kg.catalog(kind="node"))
display(
    kg.catalog(kind="edge").select(
        "dataset", "source_type", "target_type", "label", "loaded"
    )
)

kg.plot_metagraph(scope="catalog")
plt.show()


The metagraph is a type-level map. It shows which entity classes can be connected by documented edge tables. Arrow direction reflects the stored `from` and `to` convention, not biological causality.


## 3. Resolve both genes before following edges


In [ ]:
kg.load_node("gene")

display(kg.search("Galectin-3", node_type="gene", limit=20))
display(kg.search("LGALS3", node_type="gene", limit=20))
display(kg.search("NLRP3", node_type="gene", limit=20))


Substring search may return related names and synonyms. We therefore resolve both genes with explicit Ensembl identifiers. Entity resolution is an analytical decision: choosing the wrong node would change every subsequent path and metric.


In [ ]:
LGALS3_ID = "ENSG00000131981"
NLRP3_ID = "ENSG00000162711"

lgals3 = kg.entity("gene", LGALS3_ID)
nlrp3 = kg.entity("gene", NLRP3_ID)

display(lgals3.info())
display(nlrp3.info())
display(
    pl.concat(
        [
            lgals3.available_edge_types().with_columns(pl.lit("LGALS3").alias("entity")),
            nlrp3.available_edge_types().with_columns(pl.lit("NLRP3").alias("entity")),
        ],
        how="diagonal_relaxed",
    )
)


The entity handles refer to existing OptimusKG nodes. They do not copy the complete graph. Relationships are retrieved only after their edge tables are loaded and requested.


## 4. Plan and progressively load the selected evidence layers


In [ ]:
for destination in ["biological_process", "pathway", "disease", "drug"]:
    print(f"Schema route: gene -> {destination}")
    display(kg.plan_path("gene", destination, max_hops=2).show())


In [ ]:
LOAD_SPECS = [
    ("node", "biological_process"),
    ("edge", "biological_process_gene"),
    ("node", "pathway"),
    ("edge", "pathway_gene"),
    ("node", "disease"),
    ("edge", "disease_gene"),
    ("node", "drug"),
    ("edge", "drug_gene"),
]

for kind, dataset in LOAD_SPECS:
    if dataset not in kg.tables:
        if kind == "node":
            kg.load_node(dataset)
        else:
            kg.load_edge(dataset)

display(kg.loaded())
kg.plot_metagraph(scope="loaded")
plt.show()


The disease-gene table is large. It is loaded because disease evidence is part of the question, but it will be unloaded before the gene-interaction extension. This makes resource management visible rather than treating the graph as an unlimited in-memory object.


## 5. Compare typed one-hop neighbourhoods


In [ ]:
COMPARE_EDGE_TYPES = [
    "biological_process_gene",
    "pathway_gene",
    "disease_gene",
    "drug_gene",
]


def collect_neighbourhoods(entity, edge_types):
    return {
        edge_type: entity.neighbors(
            edge_types=edge_type,
            limit=None,
            exclude_self=True,
        )
        for edge_type in edge_types
    }


lgals3_neighbourhoods = collect_neighbourhoods(lgals3, COMPARE_EDGE_TYPES)
nlrp3_neighbourhoods = collect_neighbourhoods(nlrp3, COMPARE_EDGE_TYPES)


def compare_typed_neighbourhoods(left_frames, right_frames, edge_types):
    records = []
    for edge_type in edge_types:
        left_ids = set(left_frames[edge_type]["neighbor_id"].to_list())
        right_ids = set(right_frames[edge_type]["neighbor_id"].to_list())
        shared = left_ids & right_ids
        union = left_ids | right_ids
        records.append({
            "edge_type": edge_type,
            "lgals3_neighbours": len(left_ids),
            "nlrp3_neighbours": len(right_ids),
            "shared_neighbours": len(shared),
            "lgals3_only": len(left_ids - right_ids),
            "nlrp3_only": len(right_ids - left_ids),
            "jaccard_similarity": len(shared) / len(union) if union else 0.0,
        })
    return pl.DataFrame(records)


typed_comparison = compare_typed_neighbourhoods(
    lgals3_neighbourhoods,
    nlrp3_neighbourhoods,
    COMPARE_EDGE_TYPES,
)

display(typed_comparison)


In [ ]:
chart = typed_comparison.sort("edge_type")
labels = chart["edge_type"].to_list()
x = list(range(len(labels)))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    [value - width / 2 for value in x],
    [math.log1p(value) for value in chart["lgals3_neighbours"].to_list()],
    width=width,
    label="LGALS3",
    color="#6d5dfc",
)
ax.bar(
    [value + width / 2 for value in x],
    [math.log1p(value) for value in chart["nlrp3_neighbours"].to_list()],
    width=width,
    label="NLRP3",
    color="#e07a3f",
)
ax.set_xticks(x, labels, rotation=20, ha="right")
ax.set_ylabel("log(1 + unique neighbours)")
ax.set_title("Recorded one-hop evidence around LGALS3 and NLRP3")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


Raw degree measures database connectivity, not biological importance. Differences can reflect ontology structure, evidence-source coverage, publication volume or curation choices. The log scale makes small functional neighbourhoods visible beside much larger disease neighbourhoods; use the table for exact counts.


## 6. Inspect shared and unique process and pathway contexts


In [ ]:
def compare_neighbour_details(left_frame, right_frame):
    left_names = dict(zip(
        left_frame["neighbor_id"].to_list(),
        left_frame["neighbor_name"].to_list(),
    ))
    right_names = dict(zip(
        right_frame["neighbor_id"].to_list(),
        right_frame["neighbor_name"].to_list(),
    ))

    records = []
    for neighbor_id in sorted(set(left_names) | set(right_names)):
        in_left = neighbor_id in left_names
        in_right = neighbor_id in right_names
        if in_left and in_right:
            presence = "shared"
        elif in_left:
            presence = "LGALS3 only"
        else:
            presence = "NLRP3 only"
        records.append({
            "neighbor_id": neighbor_id,
            "neighbor_name": left_names.get(neighbor_id, right_names.get(neighbor_id)),
            "presence": presence,
        })
    return pl.DataFrame(records).sort("presence", "neighbor_name")


process_comparison = compare_neighbour_details(
    lgals3_neighbourhoods["biological_process_gene"],
    nlrp3_neighbourhoods["biological_process_gene"],
)
pathway_comparison = compare_neighbour_details(
    lgals3_neighbourhoods["pathway_gene"],
    nlrp3_neighbourhoods["pathway_gene"],
)

shared_process_context = process_comparison.filter(pl.col("presence") == "shared")
shared_pathway_context = pathway_comparison.filter(pl.col("presence") == "shared")

print("Shared biological-process contexts")
display(shared_process_context)
print("Shared pathway contexts")
display(shared_pathway_context)
print("Examples recorded for only one of the two genes")
display(process_comparison.filter(pl.col("presence") != "shared").head(30))


A missing shared term is not evidence that the genes are biologically unrelated. OptimusKG may omit a relationship, use a different ontology level, or inherit uneven source coverage. Conversely, a shared broad term such as `immune system process` is less informative than a specific shared process.


## 7. Create seed-derived inflammation vocabulary candidates


In [ ]:
REVIEW_PATTERN = (
    r"(?i)(inflamm\w*|immune\b|cytokines?\b|macrophages?\b|"
    r"microglia\w*|leukocytes?\b|damage\w*|wound\w*|fibro\w*|"
    r"inflammasome\w*|pyropt\w*|necropt\w*|innate\b)"
)


def candidate_terms(frame, seed_gene):
    return (
        frame
        .select("neighbor_id", "neighbor_name", "relation")
        .unique()
        .filter(pl.col("neighbor_name").str.contains(REVIEW_PATTERN))
        .with_columns(
            pl.lit(seed_gene).alias("seed_gene"),
            pl.lit(False).alias("include_in_module"),
            pl.lit("").alias("review_reason"),
            pl.lit("keyword candidate; not yet biologically reviewed").alias("status"),
        )
    )


candidate_process_terms = pl.concat(
    [
        candidate_terms(
            lgals3_neighbourhoods["biological_process_gene"], "LGALS3"
        ),
        candidate_terms(
            nlrp3_neighbourhoods["biological_process_gene"], "NLRP3"
        ),
    ],
    how="diagonal_relaxed",
).sort("neighbor_name", "seed_gene")

display(candidate_process_terms)


These are **seed-derived vocabulary candidates**, not a definition of sterile inflammation. A valid Part 2 module must be defined independently from literature and expert review and must include relevant terms that are not connected to either seed gene. Otherwise the module would be circular.


## 8. Build comparable disease-evidence tables


In [ ]:
DISEASE_PROPERTY_RENAMES = {
    "properties__evidence_score": "evidence_score",
    "properties__evidence_count": "evidence_count",
    "properties__evidence_index": "evidence_index",
    "properties__disease_specificity_index": "disease_specificity_index",
    "properties__disease_pleiotropy_index": "disease_pleiotropy_index",
    "properties__disgenet_score": "disgenet_score",
    "properties__number_of_pmids": "number_of_pmids",
    "properties__number_of_snps": "number_of_snps",
    "properties__year_initial": "year_initial",
    "properties__year_final": "year_final",
    "properties__sources__direct": "direct_sources",
}


def build_disease_landscape(entity):
    frame = entity.edge_records(
        edge_types="disease_gene",
        direction="in",
        flatten_properties=True,
        limit=None,
    )
    frame = frame.rename({
        old: new
        for old, new in DISEASE_PROPERTY_RENAMES.items()
        if old in frame.columns
    })

    # This grouping is a display aid, not an ontology classification.
    frame = frame.with_columns(
        pl.col("neighbor_id").alias("disease_id"),
        pl.col("neighbor_name").alias("disease_name"),
        pl.col("neighbor_id").str.extract(r"^([^_:]+)", 1).alias("ontology_prefix"),
        pl.when(
            pl.col("neighbor_name").str.contains(
                r"(?i)(measurement|phenotype|trait|biomarker|count|level|volume|amount|percentage)"
            )
        )
        .then(pl.lit("trait / measurement candidate"))
        .otherwise(pl.lit("disease / other candidate"))
        .alias("display_group"),
    )

    columns = [
        name for name in [
            "disease_name", "disease_id", "ontology_prefix", "display_group",
            "relation", "evidence_score", "evidence_count", "number_of_pmids",
            "number_of_snps", "disease_specificity_index",
            "disease_pleiotropy_index", "year_initial", "year_final",
            "direct_sources",
        ]
        if name in frame.columns
    ]
    result = frame.select(columns)
    sort_column = "evidence_score" if "evidence_score" in result.columns else "disease_name"
    return result.sort(
        sort_column,
        descending=(sort_column != "disease_name"),
        nulls_last=True,
    )


lgals3_disease_landscape = build_disease_landscape(lgals3)
nlrp3_disease_landscape = build_disease_landscape(nlrp3)

print("LGALS3 disease-table records")
display(lgals3_disease_landscape.head(20))
print("NLRP3 disease-table records")
display(nlrp3_disease_landscape.head(20))


In [ ]:
if {
    "evidence_score", "evidence_count"
}.issubset(lgals3_disease_landscape.columns):
    plot_data = lgals3_disease_landscape.drop_nulls(
        ["evidence_score", "evidence_count"]
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = {
        "disease / other candidate": "#6d5dfc",
        "trait / measurement candidate": "#e07a3f",
    }
    for group, color in colors.items():
        subset = plot_data.filter(pl.col("display_group") == group)
        ax.scatter(
            [math.log1p(value) for value in subset["evidence_count"].to_list()],
            subset["evidence_score"].to_list(),
            alpha=0.5,
            color=color,
            label=group,
        )
    ax.set_xlabel("log(1 + recorded evidence items)")
    ax.set_ylabel("Aggregated association evidence score")
    ax.set_title("LGALS3 disease-table evidence volume versus score")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("The selected release does not expose both evidence_score and evidence_count.")


In [ ]:
lgals3_shared_disease = lgals3_disease_landscape.select(
    "disease_id",
    "disease_name",
    pl.col("evidence_score").alias("lgals3_evidence_score"),
    pl.col("evidence_count").alias("lgals3_evidence_count"),
    pl.col("direct_sources").alias("lgals3_sources"),
)

nlrp3_shared_disease = nlrp3_disease_landscape.select(
    "disease_id",
    pl.col("evidence_score").alias("nlrp3_evidence_score"),
    pl.col("evidence_count").alias("nlrp3_evidence_count"),
    pl.col("direct_sources").alias("nlrp3_sources"),
)

shared_disease_evidence = (
    lgals3_shared_disease
    .join(nlrp3_shared_disease, on="disease_id", how="inner")
    .with_columns(
        pl.max_horizontal(
            "lgals3_evidence_score", "nlrp3_evidence_score"
        ).alias("maximum_pair_score")
    )
    .sort("maximum_pair_score", descending=True, nulls_last=True)
)

print(f"Shared disease-table concepts: {shared_disease_evidence.height:,}")
display(shared_disease_evidence.head(30))


The shared disease table identifies co-recorded contexts. It does not show that LGALS3 and NLRP3 operate through the same mechanism in those conditions. Broad ontology parents, measurements and descendants can also create apparent overlap, so exact identifiers and evidence sources should be reviewed before choosing examples for the blog.


## 9. Compare recorded compound relationships


In [ ]:
DRUG_PROPERTY_RENAMES = {
    "neighbor_id": "drug_id",
    "neighbor_name": "drug_name",
    "properties__mechanisms_of_action": "mechanisms_of_action",
    "properties__source_ids": "source_ids",
    "properties__source_urls": "source_urls",
    "properties__sources__direct": "direct_sources",
}


def build_drug_relations(entity):
    frame = entity.edge_records(
        edge_types="drug_gene",
        direction="in",
        flatten_properties=True,
        limit=None,
    )
    frame = frame.rename({
        old: new
        for old, new in DRUG_PROPERTY_RENAMES.items()
        if old in frame.columns
    })
    columns = [
        name for name in [
            "drug_name", "drug_id", "relation", "mechanisms_of_action",
            "source_ids", "source_urls", "direct_sources",
        ]
        if name in frame.columns
    ]
    return frame.select(columns).sort("drug_name")


lgals3_drug_relations = build_drug_relations(lgals3)
nlrp3_drug_relations = build_drug_relations(nlrp3)

print("LGALS3 compound relationships")
display(lgals3_drug_relations)
print("NLRP3 compound relationships")
display(nlrp3_drug_relations)


Relation types must remain distinct. A generic `TARGET` record, a biochemical inhibitor and a clinically effective treatment are not equivalent. For example, a Galectin-3 inhibitor may show target engagement without demonstrating disease efficacy.


## 10. Release the large disease layer before adding gene interactions


In [ ]:
# The two small disease landscape tables above remain available after unloading.
kg.unload("disease_gene", "disease")
gc.collect()

print("Unloaded disease_gene and disease before loading gene_gene.")
display(kg.loaded())


This step prevents the large disease layer and the gene-interaction layer from competing for memory in a small Codespace. It also demonstrates that an exploratory graph need not keep every possible table active throughout the analysis.


## 11. Audit the direct LGALS3-NLRP3 gene relationship


In [ ]:
if "gene_gene" not in kg.tables:
    kg.load_edge("gene_gene")

gene_edges = kg.tables["gene_gene"]

direct_lgals3_nlrp3 = gene_edges.filter(
    (
        (pl.col("from").cast(pl.String) == lgals3.id)
        & (pl.col("to").cast(pl.String) == nlrp3.id)
    )
    | (
        (pl.col("from").cast(pl.String) == nlrp3.id)
        & (pl.col("to").cast(pl.String) == lgals3.id)
    )
)

if direct_lgals3_nlrp3.is_empty():
    print(
        "No direct LGALS3-NLRP3 record was found in the loaded gene_gene table. "
        "This is a graph-coverage observation, not evidence of biological absence."
    )
else:
    print(f"Direct LGALS3-NLRP3 edge records: {direct_lgals3_nlrp3.height:,}")
    # Request the same records through the entity API so nested edge
    # properties are flattened into inspectable columns when supported.
    direct_edge_details = (
        lgals3.edge_records(
            edge_types="gene_gene",
            direction="both",
            flatten_properties=True,
            limit=None,
        )
        .filter(pl.col("neighbor_id") == nlrp3.id)
    )
    display(direct_edge_details)


This is an important representation test. Published experimental evidence may be present as a direct gene relationship, only through shared processes, or not represented in the selected graph release. The difference between published biology and encoded topology is itself a useful audit result.


## 12. Optional extension: compare LGALS3 interactors by process overlap


In [ ]:
self_loop_audit = gene_edges.filter(
    (pl.col("from").cast(pl.String) == lgals3.id)
    & (pl.col("to").cast(pl.String) == lgals3.id)
)

ppi_neighbours = (
    lgals3.neighbors(
        edge_types="gene_gene",
        limit=None,
        exclude_self=True,
    )
    .filter(pl.col("neighbor_type") == "gene")
    .select("neighbor_id", "neighbor_name")
    .unique("neighbor_id")
)

print(f"LGALS3 self-edge records before exclusion: {self_loop_audit.height:,}")
print(f"Other direct LGALS3 gene neighbours: {ppi_neighbours.height:,}")
display(ppi_neighbours.head(20))


In [ ]:
candidate_gene_ids = [lgals3.id] + ppi_neighbours["neighbor_id"].to_list()

membership = (
    kg.tables["biological_process_gene"]
    .filter(pl.col("to").cast(pl.String).is_in(candidate_gene_ids))
    .select(
        pl.col("to").cast(pl.String).alias("gene_id"),
        pl.col("from").cast(pl.String).alias("process_id"),
    )
    .unique()
)

process_sets = {
    gene_id: set(group["process_id"].to_list())
    for (gene_id,), group in membership.group_by("gene_id")
}

lgals3_processes = process_sets.get(lgals3.id, set())
name_map = dict(zip(
    ppi_neighbours["neighbor_id"].to_list(),
    ppi_neighbours["neighbor_name"].to_list(),
))

similarity_records = []
for gene_id in ppi_neighbours["neighbor_id"].to_list():
    other = process_sets.get(gene_id, set())
    intersection = lgals3_processes & other
    union = lgals3_processes | other
    similarity_records.append({
        "gene_id": gene_id,
        "gene_name": name_map.get(gene_id, gene_id),
        "shared_processes": len(intersection),
        "gene_processes": len(other),
        "lgals3_processes": len(lgals3_processes),
        "jaccard_similarity": len(intersection) / len(union) if union else 0.0,
    })

process_similarity = (
    pl.DataFrame(similarity_records)
    .sort(["jaccard_similarity", "shared_processes"], descending=True)
)

display(process_similarity.head(20))


Jaccard similarity is a transparent baseline, not a functional score. It treats every GO term as equally informative and can favour sparsely annotated genes. Later work can compare it with ontology-aware semantic similarity, information-content weighting and degree-matched null models.


In [ ]:
TOP_GENES = 5
MAX_SHARED_PROCESSES_PER_GENE = 3

top_gene_ids = process_similarity.head(TOP_GENES)["gene_id"].to_list()
process_rows = lgals3_neighbourhoods["biological_process_gene"]
process_name_map = dict(zip(
    process_rows["neighbor_id"].to_list(),
    process_rows["neighbor_name"].to_list(),
))

G = nx.Graph()
G.add_node(lgals3.id, label="LGALS3", node_type="gene")

for gene_id in top_gene_ids:
    gene_label = name_map.get(gene_id, gene_id)
    G.add_node(gene_id, label=gene_label, node_type="gene")
    G.add_edge(lgals3.id, gene_id, edge_type="gene interaction")

    shared = sorted(
        lgals3_processes & process_sets.get(gene_id, set()),
        key=lambda process_id: process_name_map.get(process_id, process_id),
    )
    for process_id in shared[:MAX_SHARED_PROCESSES_PER_GENE]:
        process_label = process_name_map.get(process_id, process_id)
        G.add_node(process_id, label=process_label, node_type="biological_process")
        G.add_edge(gene_id, process_id, edge_type="annotated to")
        G.add_edge(lgals3.id, process_id, edge_type="annotated to")

fig, ax = plt.subplots(figsize=(13, 8))
pos = nx.spring_layout(G, seed=9, k=1.35)

gene_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "gene"]
process_nodes = [
    n for n, d in G.nodes(data=True)
    if d["node_type"] == "biological_process"
]

nx.draw_networkx_nodes(
    G, pos, nodelist=gene_nodes, node_color="#7c5ce5", node_shape="o",
    node_size=1900, edgecolors="white", ax=ax,
)
nx.draw_networkx_nodes(
    G, pos, nodelist=process_nodes, node_color="#f2a93b", node_shape="s",
    node_size=1700, edgecolors="white", ax=ax,
)

interaction_edges = [
    (u, v) for u, v, data in G.edges(data=True)
    if data["edge_type"] == "gene interaction"
]
annotation_edges = [
    (u, v) for u, v, data in G.edges(data=True)
    if data["edge_type"] == "annotated to"
]

nx.draw_networkx_edges(
    G, pos, edgelist=interaction_edges,
    edge_color="#5b4acb", width=2.0, ax=ax,
)
nx.draw_networkx_edges(
    G, pos, edgelist=annotation_edges,
    edge_color="#9aa4b2", width=1.1, style="dashed", ax=ax,
)

labels = {
    node: (data["label"][:31] + "..." if len(data["label"]) > 34 else data["label"])
    for node, data in G.nodes(data=True)
}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)

ax.set_title("Representative LGALS3 interaction-process subgraph")
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#7c5ce5",
           markersize=10, label="Gene"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="#f2a93b",
           markersize=10, label="Biological process"),
    Line2D([0], [0], color="#5b4acb", lw=2, label="Gene interaction"),
    Line2D([0], [0], color="#9aa4b2", lw=1.1, linestyle="--",
           label="Recorded process annotation"),
], loc="upper left", frameon=False, fontsize=8)
ax.axis("off")
plt.tight_layout()
plt.show()


The network is deliberately selective. It is a communication view of a documented selection rule, not the complete LGALS3 network. The tables remain the evidence record.


## 13. Summarise what the tutorial established


In [ ]:
def comparison_value(edge_type, column):
    values = typed_comparison.filter(
        pl.col("edge_type") == edge_type
    ).get_column(column)
    return values[0] if len(values) else None


tutorial_summary = pl.DataFrame({
    "measure": [
        "OptimusKG release",
        "LGALS3 biological-process neighbours",
        "NLRP3 biological-process neighbours",
        "Shared biological-process neighbours",
        "Shared pathway neighbours",
        "Shared disease-table concepts",
        "LGALS3 recorded compound relations",
        "NLRP3 recorded compound relations",
        "Direct LGALS3-NLRP3 gene edge records",
        "Seed-derived inflammation vocabulary rows",
    ],
    # Values are strings so counts and identifiers can share one audit column.
    "value": [
        str(OPTIMUSKG_DOI),
        str(comparison_value("biological_process_gene", "lgals3_neighbours")),
        str(comparison_value("biological_process_gene", "nlrp3_neighbours")),
        str(comparison_value("biological_process_gene", "shared_neighbours")),
        str(comparison_value("pathway_gene", "shared_neighbours")),
        str(shared_disease_evidence.height),
        str(lgals3_drug_relations.height),
        str(nlrp3_drug_relations.height),
        str(direct_lgals3_nlrp3.height),
        str(candidate_process_terms.height),
    ],
})

display(tutorial_summary)


### How to interpret the final comparison

Use the output to distinguish three outcomes:

1. **Graph and literature agree.** Shared or direct relationships reproduce a documented context.
2. **Literature reports a relationship that the graph does not encode.** This is a coverage, schema or release-timing question.
3. **The graph exposes an indirect connection not obvious from the starting papers.** This becomes a candidate for targeted literature review or experimental validation.

The useful result is not simply that one gene has more edges. It is knowing which typed evidence is present, which is absent, how much is shared, and what provenance supports it.


## 14. Optional CSV snapshots


In [ ]:
SAVE_CSV_SNAPSHOTS = False

analysis_tables = {
    "tutorial_summary": tutorial_summary,
    "lgals3_nlrp3_typed_neighbourhood_comparison": typed_comparison,
    "lgals3_nlrp3_process_contexts": process_comparison,
    "lgals3_nlrp3_pathway_contexts": pathway_comparison,
    "lgals3_nlrp3_shared_disease_evidence": shared_disease_evidence,
    "seed_derived_inflammation_process_candidates": candidate_process_terms,
    "lgals3_disease_landscape": lgals3_disease_landscape,
    "nlrp3_disease_landscape": nlrp3_disease_landscape,
    "lgals3_drug_relations": lgals3_drug_relations,
    "nlrp3_drug_relations": nlrp3_drug_relations,
    "lgals3_interactor_process_similarity": process_similarity,
}

if SAVE_CSV_SNAPSHOTS:
    for name, frame in analysis_tables.items():
        kg.export_table(frame, OUTPUT_DIR / f"{name}.csv")
        print(f"Saved {name}.csv ({frame.height:,} rows)")
    print("CSV snapshots saved to:", OUTPUT_DIR)
else:
    print(
        "CSV export is disabled. Set SAVE_CSV_SNAPSHOTS = True "
        "after reviewing the tables."
    )


The CSV files are optional hand-off tables for review, spreadsheet tools and later visualisation. This tutorial intentionally does not create a database. Database packaging belongs in the later sterile-inflammation workflow, once the analytical module and table relationships have been reviewed.


## 15. Student reflection questions

1. Why was exact Ensembl resolution necessary for both genes?
2. Which relationship type produced the largest neighbourhood for each entity?
3. Which differences may reflect biology, and which may reflect source coverage?
4. Which biological processes and pathways were shared?
5. Were the shared terms specific enough to support a biological interpretation?
6. Did OptimusKG contain a direct LGALS3-NLRP3 gene relationship?
7. If the direct relationship was absent, how would you distinguish a graph gap from biological absence?
8. Which compound edges described a directional mechanism and which were generic target records?
9. How did evidence score differ from evidence count?
10. What must be independently defined before studying a sterile-inflammation module?
11. What would ontology-aware similarity or a degree-matched background add?
12. What experimental, transcriptomic, tissue or temporal evidence would be required before describing a path as causal?

### Suggested conclusion template

> In the pinned OptimusKG release, LGALS3 and NLRP3 showed **[different/similar]** recorded neighbourhoods across disease, compound and functional relationship types. They shared **[number]** biological-process contexts and **[number]** pathway contexts. The graph **[did/did not]** encode a direct gene relationship, while selected literature describes context-specific Galectin-3/NLRP3 relationships in **[systems]**. The discrepancy highlights that a knowledge graph is a structured map of available evidence rather than a complete model of biology. These results support a separately defined sterile-inflammation study using reviewed terms, topology-aware backgrounds and external validation.


## References and future direction

### Biology

- [DAMP sensing and sterile inflammation](https://www.nature.com/articles/s41577-024-01027-3)
- [NLRP3 inflammasome activation and regulation](https://pmc.ncbi.nlm.nih.gov/articles/PMC7807242/)
- [Galectin-3 regulates inflammasome activation in cholestatic liver injury](https://pmc.ncbi.nlm.nih.gov/articles/PMC5102125/)
- [Galectin-3 in microglia-mediated inflammation in a Huntington's disease model](https://pmc.ncbi.nlm.nih.gov/articles/PMC6677843/)
- [Belapectin phase 2b trial](https://pubmed.ncbi.nlm.nih.gov/31812510/)
- [Inhaled TD139/olitigaltin phase 1/2a study](https://pubmed.ncbi.nlm.nih.gov/33214209/)

### Knowledge-graph precedents

- [KnetMiner: evidence-based gene discovery](https://pmc.ncbi.nlm.nih.gov/articles/PMC8244015/)
- [ROBOKOP: biomedical query graphs and mechanistic hypotheses](https://pubmed.ncbi.nlm.nih.gov/?term=ROBOKOP%3A+an+abstraction+layer+and+user+interface+for+knowledge+graphs)
- [Project Rephetio: typed metapaths and degree-weighted path counts](https://elifesciences.org/articles/26726)

### Recommended Part 2 progression

1. define a sterile-inflammation module independently from LGALS3 and NLRP3;
2. review terms by damage, sensing, amplification, recruitment, resolution and fibrosis phases;
3. compare genes and compounds against degree- and annotation-matched backgrounds;
4. examine explicit typed paths and their provenance;
5. only then test random walks, semantic similarity, embeddings or link prediction;
6. validate prioritized paths against literature, tissue-specific data or perturbational evidence.
